# Diffusion Vocal + Crackle Retool

This notebook launches a diffusion fine-tune specifically aimed at the two main failure modes we are hearing:

- vocals that sound unstable or unnatural
- crackly / static-like high-frequency artifacts that compound in long-form rollout

What changed in the trainer:
- a vocal-band stability loss to keep the core vocal region from getting shredded
- a crackle penalty that pushes down excessive frame-to-frame HF jitter
- the existing adjacent-chunk continuity loss is kept, so the model still sees long-form-relevant structure during training

This notebook uses a practical capped regime so iteration is feasible.

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib
import json
import os
import subprocess
import sys

def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / 'lab 3.1').exists() and (p / 'README.md').exists():
            return p
    raise RuntimeError('Could not locate repo root from current notebook cwd.')

REPO = find_repo_root()
SCRIPT_DIR = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import diffusion_longform_retool_train as retool
importlib.reload(retool)

print(REPO)

In [ ]:
TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = REPO / 'lab 3.1' / 'outputs' / 'diffusion_vocal_crackle_retool' / f'run_{TAG}'
CACHE_DIR = REPO / 'saves2' / 'lab3_diffusion' / 'run_d001' / 'cache'
BASE_CHECKPOINT = REPO / 'saves2' / 'lab3_diffusion' / 'run_d002' / 'checkpoints' / 'best.pt'

RUN_TRAIN = False

TRAIN_CFG = {
    'epochs': 6,
    'batch_size': 1,
    'grad_accum': 2,
    'max_frames': 256,
    'lr': 7e-5,
    'identity_weight': 1.0,
    'style_weight': 1.7,
    'anchor_weight': 0.45,
    'envelope_weight': 0.25,
    'continuity_weight': 0.75,
    'hf_penalty_weight': 0.22,
    'vocal_weight': 0.55,
    'crackle_weight': 0.35,
    'anchor_bins': 40,
    'hf_start_bin': 56,
    'vocal_start_bin': 10,
    'vocal_end_bin': 42,
    'overlap_frames': 40,
    'hf_margin': 0.05,
    'crackle_margin': 0.012,
    'style_every_steps': 2,
    'style_batch_splits': 1,
    'max_batches_per_epoch': 8000,
    'monitor_steps': 50,
    'device': 'auto',
}

print('OUT_DIR =', OUT_DIR)
print('BASE_CHECKPOINT =', BASE_CHECKPOINT)
print(json.dumps(TRAIN_CFG, indent=2))

In [ ]:
train_cmd = [
    sys.executable,
    str(SCRIPT_DIR / 'diffusion_longform_retool_train.py'),
    '--cache-dir', str(CACHE_DIR),
    '--out-dir', str(OUT_DIR),
    '--bootstrap-checkpoint', str(BASE_CHECKPOINT),
]
for key, value in TRAIN_CFG.items():
    train_cmd += ['--' + key.replace('_', '-'), str(value)]

print(' '.join(map(str, train_cmd)))
train_log = OUT_DIR / 'train.log'

if RUN_TRAIN:
    env = dict(os.environ)
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with train_log.open('w', encoding='utf-8', errors='replace') as log:
        proc = subprocess.Popen(train_cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', env=env)
        ret = proc.wait()
    print(train_log)
    print(train_log.read_text(encoding='utf-8', errors='replace')[-12000:])
    if ret != 0:
        raise RuntimeError(f'Training failed with exit code {ret}. See {train_log}')
else:
    print('Set RUN_TRAIN = True to launch the vocal/crackle diffusion retool run.')

In [ ]:
history_path = OUT_DIR / 'v2_history.json'
if history_path.exists():
    import pandas as pd
    hist = pd.read_json(history_path)
    display(hist.tail(10))
else:
    print('No history yet for this tag.')